### Add mutatios for robustness testing

In [1]:
def default_params(): 
    return {
        'bert_model': 'microsoft/codebert-base-mlm',
        'cache_dir': '/workspaces/CodeSmells/datax/hugging_face_cache',
        'dataset_path' : '/workspaces/CodeSmells/semeru-datasets/code_smells',
        'sampling_size' : 500
    }
params = default_params()


### Imports

In [2]:
import torch
import gc
import json
import subprocess
import os
import glob

In [3]:
import pandas as pd
import numpy as np
from CodeSmells import semantic_preserving_transformations as trans

2025-02-11 20:32:14.882138: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1739305934.900695 3725380 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1739305934.906309 3725380 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-11 20:32:14.925392: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [4]:
from transformers import logging

logging.set_verbosity_error()


### Read Dataframe

In [5]:
dataset_df = pd.read_json(f"{params['dataset_path']}/curated_{params['sampling_size']}.json")

In [6]:
dataset_df = dataset_df[:20]

### Add mutations

In [7]:
def run_pylint_analysis(code):
    temp_file_path = f"{params['cache_dir']}/pylint/temp.py"
     # Save the code to a temporary file with UTF-8 encoding
    with open(temp_file_path, 'w', encoding='utf-8') as temp_file:
        temp_file.write(code)
    command = [
        'pylint',
        temp_file_path,
        '--output-format=json'
    ]
    result = subprocess.run(command, capture_output=True, text=True)
    full_analysis = json.loads(result.stdout) if result.stdout else []

    # Read the code again to extract lines, using UTF-8 encoding
    with open(temp_file_path, 'r', encoding='utf-8') as file:
        lines = file.readlines()
    os.remove(temp_file_path)  # Clean up the temporary file

    simplified_analysis = []
    for issue in full_analysis:
        line = int(issue['line']) if issue['line'] is not None else 0
        end_line = int(issue.get('endLine', line)) if issue.get('endLine', line) is not None else line
        start_line = line - 1
        end_line = end_line - 1

        issue_code = ''.join(lines[start_line:end_line + 1]).strip() if start_line >= 0 and end_line >= 0 and start_line <= end_line else ""

        simplified_analysis.append({
            'code' : code,
            'msg_id': issue['message-id'],
            'line': issue['line'],
            'column': issue['column'],
            'end_line': issue.get('endLine', issue['line']),
            'end_column': issue['endColumn'],
            'code_smell': issue_code
        })

    return simplified_analysis

In [8]:
def compute_end_column(smell):
    if pd.notna(smell['end_column']):
        return smell['end_column']
    code_lines = smell['code'].splitlines()
    
    # Validate that end_line is within the bounds of code_lines
    if smell['end_line'] < 0 or smell['end_line'] >= len(code_lines):
       raise IndexError("end_line is out of range.")
    
    return len(code_lines[smell['end_line']])


In [9]:
def extract_substring(code_string, start_line, start_column, end_line, end_column):
    # Split the string into individual lines.
    lines = code_string.splitlines()
    # Validate the provided indices.
    if start_line < 0 or start_line >= len(lines):
        raise IndexError("start_line is out of range.")
    if end_line < 0 or end_line >= len(lines):
        raise IndexError("end_line is out of range.")
    # Case when the substring is within a single line.
    if start_line == end_line:
        return lines[start_line][start_column:end_column]
    # Extract parts from multiple lines.
    # 1. Extract from the start line starting at start_column.
    extracted_lines = [lines[start_line][start_column:]]
    # 2. Add all the lines between the start and end lines (if any).
    for line in lines[start_line + 1 : end_line]:
        extracted_lines.append(line)
    # 3. Extract from the end line up to end_column.
    extracted_lines.append(lines[end_line][:end_column])
    # Join the parts with newline characters.
    return "\n".join(extracted_lines)

In [10]:
def find_first_matching_object(row, transformation_column:str):
    return next((obj for obj in row[transformation_column] if obj.get('msg_id') == row['s_msg_id'] and obj.get('line') == row['s_line']), None)


In [11]:
def fix_smell_pos_values(smell):
    smell['line'] -= 1
    if pd.isna(smell['end_line']):
        smell['end_line'] = smell['line']
    else: 
        smell['end_line'] -= 1
    smell['end_column'] = compute_end_column(smell)
    smell['code_smell'] = extract_substring(smell['code'], smell['line'], smell['column'], smell['end_line'], smell['end_column'])
    return smell
        

In [12]:
def add_transformation(df, transformation_name, transformation_function, filter_unique):
    df = df.copy()
    df[transformation_name] = df.apply(lambda row: [fix_smell_pos_values(smell) for smell in run_pylint_analysis(transformation_function(row['code']))], axis=1)
    df[transformation_name] = df.apply(lambda row: find_first_matching_object(row, transformation_name), axis=1)
    ###filter by change in code
    if filter_unique: df[transformation_name] = df.apply(lambda row: row[transformation_name] if (row[transformation_name] and row[transformation_name]['code_smell']!= row['s_code']) else None, axis=1)
    df = df.reset_index(drop=True)
    return df


In [13]:
def execute_rename_variable_2(code:str):
    return trans.rename_variable_2(code, params['bert_model'], params['cache_dir'])

In [14]:
dataset_df = add_transformation(dataset_df, 'RenameVariable-1', trans.rename_variable_1, False)
dataset_df = add_transformation(dataset_df, 'RenameVariable-2', execute_rename_variable_2, False)
dataset_df = add_transformation(dataset_df, 'Add2Equal', trans.add_2_equal, False)
dataset_df = add_transformation(dataset_df, 'SwitchEqualExp', trans.switch_equal_exp, False)
dataset_df = add_transformation(dataset_df, 'InfixDividing', trans.infix_dividing, False)
dataset_df = add_transformation(dataset_df, 'SwitchRelation', trans.switch_relation, False)

/usr/local/lib/python3.11/dist-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


In [15]:
dataset_df

,id,commit_id,repo,path,file_name,fun_name,commit_message,code,url,language,...,s_end_line,s_end_column,s_code,category,RenameVariable-1,RenameVariable-2,Add2Equal,SwitchEqualExp,InfixDividing,SwitchRelation
0,261282,e41753ebd57c44ae91b389f190c43ddc0b384a75,scikit-learn,sklearn/linear_model/tests/test_coordinate_des...,test_coordinate_descent.py,test_sample_weight_invariance,MAINT Clean deprecation for 1.2: normalize in ...,def test_sample_weight_invariance(estimator):\...,https://github.com/scikit-learn/scikit-learn.git,Python,...,0,33,def test_sample_weight_invariance,Refactor,None,None,{'code': 'def test_sample_weight_invariance(es...,{'code': 'def test_sample_weight_invariance(es...,{'code': 'def test_sample_weight_invariance(es...,{'code': 'def test_sample_weight_invariance(es...
1,152612,8deae077004f0332ca607fc3a5d568b1a4705bec,stable-diffusion-webui,modules/scunet_model.py,scunet_model.py,__init__,Add ScuNET DeNoiser/Upscaler\n\nQ&D Implementa...,"def __init__(self, dirname):\n self.nam...",https://github.com/AUTOMATIC1111/stable-diffus...,Python,...,16,54,name = modelloader.friendly_na...,Warning,"{'code': 'def _(s, d): s.n = ""ScuNET"" ...","{'code': 'def __init__(d, path): d.nam...","{'code': 'def __init__(self, dirname): ...","{'code': 'def __init__(self, dirname): ...","{'code': 'def __init__(self, dirname): ...","{'code': 'def __init__(self, dirname): ..."
2,264725,d858eceb387f65ec96d297def940205cddee7bf3,netbox,netbox/dcim/forms/models.py,models.py,clean,Fix pep8,def clean(self):\n super().clean()\n\n ...,https://github.com/netbox-community/netbox.git,Python,...,36,71,existing_item = installed_comp...,Warning,{'code': 'def c(s): s().c() r...,{'code': 'def clean(d): super().clean(...,{'code': 'def clean(self): super().cle...,{'code': 'def clean(self): super().cle...,{'code': 'def clean(self): super().cle...,{'code': 'def clean(self): super().cle...
3,15607,5697ef2f1872813eafd59f27aa7d650598492f9d,ccxt,python/ccxt/huobi.py,huobi.py,fetch_currencies,1.67.46\n\n[ci skip],"def fetch_currencies(self, params={}):\n ...",https://github.com/ccxt/ccxt.git,Python,...,64,27,minWithdraw,Convention,None,None,"{'code': 'def fetch_currencies(self, params={}...","{'code': 'def fetch_currencies(self, params={}...","{'code': 'def fetch_currencies(self, params={}...","{'code': 'def fetch_currencies(self, params={}..."
4,127125,753fad9cadc395fcb6d8d8ecafa1f808a6343628,ray,rllib/algorithms/dt/tests/test_segmentation_bu...,test_segmentation_buffer.py,test_add,[RLlib] Add Segmentation Buffer for DT (#27829),def test_add(self):\n \n for buf...,https://github.com/ray-project/ray.git,Python,...,4,27,max_ep_len = 10,Warning,{'code': 'def t(s): for b in ...,{'code': 'def test_add(d): fo...,{'code': 'def test_add(self): ...,{'code': 'def test_add(self): ...,{'code': 'def test_add(self): ...,{'code': 'def test_add(self): ...
5,166553,46bcf3740b38339f62b94e66ec29537a28a17140,pandas,pandas/tests/indexing/test_indexing.py,test_indexing.py,test_astype_assignment_full_replacements,"DEPR: df.iloc[:, foo] = bar attempt to set inp...",def test_astype_assignment_full_replacements(s...,https://github.com/pandas-dev/pandas.git,Python,...,2,51,"df = DataFrame({""A"": [1.0, 2.0, 3.0, 4...",Warning,{'code': 'def t(s): # full replacement...,{'code': 'def test_astype_assignment_full_repl...,{'code': 'def test_astype_assignment_full_repl...,{'code': 'def test_astype_assignment_full_repl...,{'code': 'def test_astype_assignment_full_repl...,{'code': 'def test_astype_assignment_full_repl...
6,266937,f96a661adaf627764fc3527cab1ca4d829d69ea1,ansible,lib/ansible/cli/galaxy.py,galaxy.py,add_verify_options,ansible-galaxy - add configuration options for...,"def add_verify_options(self, parser, parents=N...",https://github.com/ansible/ansible.git,Python,...,8,117,"verify_parser.add_argument('-i', '--ig...",Convention,None,"{'code': 'def add_verify_options(d, _, parent=...","{'code': 'def add_verify_options(self, parser,...","{'code': 'def add_verify_options(self, parser

In [16]:
print(dataset_df.loc[15]['s_code'])
print(dataset_df.loc[15]['RenameVariable-1']['code_smell'])
print(dataset_df.loc[15]['RenameVariable-2']['code_smell'])
print(dataset_df.loc[15]['Add2Equal']['code_smell'])


C
N
N
N
